# AI-sfa 컬러 이펙트 LoRA 3차 — 회색 연기 태그 + 인물 자리 마스크 손실 (Kaggle T4)
2차 컬러 LoRA는 강도·부정어와 상관없이 폭발을 회색 연기로 그렸다(2026-10-26 프롬프트 조합 시험 2회차). 자료에서 두 가지를 고쳤다(저장소 tools/lora_v3_data.py).
- **`grey smoke` 태그**: 그림의 회색 연기 면적이 25% 이상이면 붙임(185장). 생성 때 부정어 `grey smoke`로 연기만 뺄 수 있게.
- **마스크 손실**: 인물 자리를 검게 칠하던 판(255장)을 원본 + 인물 자리 알파 0인 PNG로 바꾸고 `--masked_loss`로 그 자리는 학습에서 뺌('검은 덩어리' 학습 방지).
- 컬러만 학습(임팩트 LoRA는 2차 결과가 좋아 그대로). 4바퀴, 바퀴마다 저장. 약 4시간.
- 설정: **GPU T4 x2**, **Internet On** → **Save Version → Save & Run All (Commit)**.
- 끝나면 Output의 `lora_v3_result.zip`(LoRA + 확인 그림 + grid_color.jpg)을 Drive 공유 링크로 알려 준다.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /tmp /kaggle/working | tail -2
!free -g | head -2

In [ ]:
# 1) 학습 환경: 서버에서 불러오기까지 확인한 조합을 별도 환경(/tmp/env)에 그대로 깐다(Kaggle 기본 패키지와 섞이지 않게)
!pip -q install uv
!uv venv -q /tmp/env --python 3.10
!uv pip install -q --python /tmp/env/bin/python torch==2.4.1 torchvision==0.19.1 --index-url https://download.pytorch.org/whl/cu124
!git clone -q --depth 1 --branch v0.9.1 https://github.com/kohya-ss/sd-scripts.git /tmp/sd-scripts
!cd /tmp/sd-scripts && uv pip install -q --python /tmp/env/bin/python -r requirements.txt
!uv pip install -q --python /tmp/env/bin/python lycoris-lora==3.1.1 "numpy==1.26.4"
!cd /tmp/sd-scripts && /tmp/env/bin/python -c "import torch, library, lycoris, bitsandbytes, cv2; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"

In [ ]:
# 2) 학습 자료 3차(Drive 공유 링크)와 기본 모델(Hugging Face)
!curl -sSL -o /tmp/data.zip "https://drive.usercontent.google.com/download?id=1pyFFQ2r08DRLGy0OWso11QsyvFR4jG1F&export=download&confirm=t"
!mkdir -p /tmp/data && cd /tmp/data && unzip -q -o /tmp/data.zip && for d in color color_muted; do echo $d jpg $(ls $d/*.jpg | wc -l) png $(ls $d/*.png | wc -l); done
!grep -l "grey smoke" /tmp/data/color*/*.txt | wc -l
!curl -sSL -C - -o /tmp/ilxl.safetensors https://huggingface.co/OnomaAIResearch/Illustrious-XL-v1.0/resolve/main/Illustrious-XL-v1.0.safetensors
!ls -la /tmp/ilxl.safetensors


In [ ]:
# 3) 자료 설정(캡션 섞기 끔: 텍스트 인코더 출력을 미리 계산하므로). alpha_mask: PNG 알파 0 = 학습 안 함(알파 없는 jpg는 전체 학습)
import os, glob
T = '''[general]
caption_extension = ".txt"
shuffle_caption = false

[[datasets]]
resolution = 1024
batch_size = 1
enable_bucket = true
min_bucket_reso = 640
max_bucket_reso = 1536
bucket_reso_steps = 64
'''
open('/tmp/color.toml','w').write(T + '''
  [[datasets.subsets]]
  image_dir = "/tmp/data/color"
  num_repeats = 2
  alpha_mask = true

  [[datasets.subsets]]
  image_dir = "/tmp/data/color_muted"
  num_repeats = 1
  alpha_mask = true
''')
print(open('/tmp/color.toml').read())


In [ ]:
# 4) 학습: 컬러만(약 4시간), 마스크 손실. T4: 기본 모델 fp8 + fp16 계산, VAE fp32
!mkdir -p /kaggle/working/out
def train(name, toml, epochs, every):
    cmd = f"""cd /tmp/sd-scripts && /tmp/env/bin/python sdxl_train_network.py \
  --pretrained_model_name_or_path /tmp/ilxl.safetensors --dataset_config {toml} \
  --output_dir /kaggle/working/out --output_name {name} \
  --network_module lycoris.kohya --network_dim 32 --network_alpha 8 \
  --network_args algo=locon preset=full conv_dim=16 conv_alpha=4 \
  --network_train_unet_only --unet_lr 4e-4 \
  --optimizer_type AdamW8bit --lr_scheduler cosine --lr_warmup_steps 100 \
  --max_train_epochs {epochs} --save_every_n_epochs {every} --save_model_as safetensors \
  --mixed_precision fp16 --save_precision fp16 --fp8_base --no_half_vae --sdpa --gradient_checkpointing \
  --cache_latents --cache_latents_to_disk --cache_text_encoder_outputs --cache_text_encoder_outputs_to_disk \
  --min_snr_gamma 5 --masked_loss --max_data_loader_n_workers 2 --seed 42"""
    r = os.system(cmd)
    print(name, 'exit', r)
train('nkmr_fx_color3', '/tmp/color.toml', 4, 1)
!ls -la /kaggle/working/out

In [ ]:
# 5) 확인 그림: 기본 모델 / 각 바퀴, 시드 2개. 핵심 = 폭발 글에서 부정어 grey smoke가 연기만 빼는지
N0 = "lowres, worst quality, text, watermark"
NS = N0 + ", grey smoke"
EXP = "nkmr_fx, vivid colors, anime screencap, explosion, fire, radial, white core orange rim, layered cel shading, no outline, sharp tips, peak, large effect, no humans, simple background, wide shot"
color_p = [
 ("nkmr_fx, anime screencap, explosion, smoke, billowing mass, white core orange rim, layered cel shading, no outline, sharp tips, expanding, large effect, no humans, simple background, wide shot", N0),
 (EXP, N0),
 (EXP, NS),
 ("nkmr_fx, vivid colors, anime screencap, fire, flames, pillar, white core orange rim, layered cel shading, sharp tips, expanding, large effect, no humans, city, night, from below", N0),
 ("nkmr_fx, anime screencap, lightning, embers, streaks, particles, white core blue rim, no outline, sharp tips, rapid expansion, no humans, city, wide shot", N0),
 ("nkmr_fx, grey smoke, anime screencap, debris, dust cloud, billowing mass, flat color, simple cel shading, dispersing, flying, no humans, rocky ground, wide shot", N0),
]
def sample(tag, prompts, ckpts, seeds=(1234, 777)):
    open(f'/tmp/p_{tag}.txt','w').write("\n".join(f"{p} --n {n} --w 1344 --h 768 --s 28 --l 6 --d {sd}" for sd in seeds for p, n in prompts))
    for name, c in [("base", None)] + [(os.path.basename(c)[:-12], c) for c in ckpts]:
        out = f"/kaggle/working/samples_{tag}/{name}"
        os.makedirs(out, exist_ok=True)
        net = f"--network_module lycoris.kohya --network_weights {c} --network_mul 1.0 --network_merge" if c else ""
        os.system(f"cd /tmp/sd-scripts && /tmp/env/bin/python sdxl_gen_img.py --ckpt /tmp/ilxl.safetensors --outdir {out} --from_file /tmp/p_{tag}.txt --images_per_prompt 1 --fp16 --no_half_vae --sdpa --sampler euler_a {net} > {out}/log.txt 2>&1")
        print(tag, name, len(glob.glob(out + '/*.png')), 'images')
sample("color", color_p, sorted(glob.glob('/kaggle/working/out/nkmr_fx_color3*.safetensors')))


In [ ]:
# 6) 격자(줄 = 기본/바퀴, 칸 = 글) + 묶기
from PIL import Image, ImageDraw
def grid(tag):
    rows = sorted(glob.glob(f'/kaggle/working/samples_{tag}/*/'), key=lambda r: (r.rstrip('/').split('/')[-1] != 'base', r))
    rows = [(r.rstrip('/').split('/')[-1], sorted(glob.glob(r + '*.png'))) for r in rows]
    rows = [(n, ims) for n, ims in rows if ims]
    if not rows: return
    W = Image.new('RGB', (448 * max(len(i) for _, i in rows), 256 * len(rows)), 'black')
    d = ImageDraw.Draw(W)
    for y, (n, ims) in enumerate(rows):
        for x, p in enumerate(ims):
            W.paste(Image.open(p).resize((448, 256)), (448 * x, 256 * y))
        d.rectangle((0, 256 * y, 190, 256 * y + 20), fill='black'); d.text((4, 256 * y + 4), n, fill='white')
    W.save(f'/kaggle/working/grid_{tag}.jpg', quality=88)
    print(tag, [(n, len(i)) for n, i in rows])
grid("color")
!cd /kaggle/working && zip -q -r lora_v3_result.zip out samples_color grid_color.jpg && ls -la /kaggle/working